In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import requests,struct,zlib,json,gzip,hashlib
from datetime import datetime,timezone
OUT=Path('/content/drive/MyDrive/Allinone/Exercise_Quality_95_Target_20261028')
BASE='https://softwarecenter.qualcomm.com/api/download/software/dataset/AIDataset/Qualcomm_Exercise_Video_Dataset/'
URL=BASE+'QEVD-FIT-300k-Part-4/QEVD-FIT-300k-Part-4.zip'
report={'status':'pending','source_url':URL,'raw_videos_downloaded':0,'utc':datetime.now(timezone.utc).isoformat()}
def get_range(url,a,b=None,suffix=False):
 header=f'bytes=-{a}' if suffix else f'bytes={a}-{b}'
 with requests.get(url,headers={'Range':header,'Accept-Encoding':'identity'},stream=True,timeout=(20,120)) as r:
  if r.status_code!=206:raise RuntimeError(f'Range not supported: HTTP{r.status_code}, type={r.headers.get("Content-Type")}')
  data=bytearray()
  for chunk in r.iter_content(1024*1024):
   data.extend(chunk)
   if len(data)>64*1024**2:raise RuntimeError('Bounded metadata request exceeded64MiB')
  return bytes(data),r.headers.get('Content-Range')
try:
 tail,cr=get_range(URL,65536,suffix=True);report['content_range']=cr
 i=tail.rfind(b'PK\x05\x06')
 if i<0:raise RuntimeError('No ZIP end record')
 e=struct.unpack_from('<4s4H2IH',tail,i);disk,cd_disk,cd_size,cd_offset=e[1],e[2],e[5],e[6]
 if cd_size==0xffffffff or cd_offset==0xffffffff:
  j=tail.rfind(b'PK\x06\x06')
  if j<0:raise RuntimeError('ZIP64 metadata outside bounded tail')
  q=struct.unpack_from('<4sQ2H2I4Q',tail,j);disk,cd_disk,cd_size,cd_offset=q[4],q[5],q[8],q[9]
 report.update(last_disk=disk,central_disk=cd_disk,central_bytes=cd_size,central_offset=cd_offset)
 if disk!=cd_disk or cd_size>64*1024**2:raise RuntimeError('Central directory spans disks or exceeds metadata budget')
 cd,_=get_range(URL,cd_offset,cd_offset+cd_size-1);entries=[];pos=0
 while cd[pos:pos+4]==b'PK\x01\x02':
  h=struct.unpack_from('<4s6H3I5H2I',cd,pos);fn,el,cl=h[10:13];name=cd[pos+46:pos+46+fn].decode('utf-8','replace');entry={'name':name,'method':h[4],'compressed':h[8],'uncompressed':h[9],'disk':h[13],'offset':h[16]};extra=cd[pos+46+fn:pos+46+fn+el];k=0
  while k+4<=len(extra):
   tag,sz=struct.unpack_from('<HH',extra,k);v=extra[k+4:k+4+sz];k+=4+sz
   if tag==1:
    q=0
    for field,width in [('uncompressed',8),('compressed',8),('offset',8),('disk',4)]:
     if entry[field]==(0xffffffff if width==8 else 0xffff):entry[field]=int.from_bytes(v[q:q+width],'little');q+=width
  entries.append(entry);pos+=46+fn+el+cl
 report['central_entries']=len(entries);annotations=[x for x in entries if 'label' in x['name'].lower() and x['name'].lower().endswith('.json')];report['annotation_entries']=annotations
 if not annotations:raise RuntimeError('No label JSON in last-part directory')
 entry=next((x for x in annotations if 'fine_grained' in x['name']),annotations[0]);suffix=['z01','z02','zip'][entry['disk']];parturl=BASE+f'QEVD-FIT-300k-Part-4/QEVD-FIT-300k-Part-4.{suffix}'
 header,_=get_range(parturl,entry['offset'],entry['offset']+65535)
 if header[:4]!=b'PK\x03\x04':raise RuntimeError('Local header offset does not match split ZIP')
 local=struct.unpack_from('<4s5H3I2H',header);start=entry['offset']+30+local[-2]+local[-1]
 payload,_=get_range(parturl,start,start+entry['compressed']-1)
 raw=zlib.decompress(payload,-15) if entry['method']==8 else payload
 data=json.loads(raw);rows=data if isinstance(data,list) else next((v for v in data.values() if isinstance(v,list)),[]);report.update(annotation_rows=len(rows),annotation_examples=rows[:2])
 workers=requests.get(BASE+'QEVD-FIT-300k-Additional-Labels/fine_grained_labels_with_worker_ids.json',timeout=(20,120)).json();wm={Path(x['video_path']).name:x.get('worker_id') for x in workers}
 selected=[];labels={}
 terms=['push','squat','bicep','curl','shoulder','overhead','row','tricep','side bend','side-bend']
 for x in rows:
  ll=x.get('labels',[])
  if isinstance(ll,str):ll=[ll]
  for label in ll:labels[label]=labels.get(label,0)+1
  if any(t in json.dumps(ll).lower() for t in terms):
   selected.append({'video_path':x.get('video_path'),'labels':ll,'labels_descriptive':x.get('labels_descriptive'),'worker_id':wm.get(Path(x.get('video_path','')).name),'publisher_split':x.get('split')})
 report.update(status='annotation_metadata_acquired',selected_rows=len(selected),selected_subjects=len({x['worker_id'] for x in selected if x['worker_id'] is not None}),selected_missing_subject=sum(x['worker_id'] is None for x in selected),label_counts=labels)
 with gzip.open(OUT/'qevd_candidate_annotations.json.gz','wt',encoding='utf-8') as f:json.dump(selected,f)
 # Compact zip-member index supports future bounded per-video extraction without full archives.
 with gzip.open(OUT/'qevd_part4_member_index.json.gz','wt',encoding='utf-8') as f:json.dump(entries,f)
except Exception as e:report.update(status='annotation_access_failed',error=repr(e))
(OUT/'qevd_annotation_probe.json').write_text(json.dumps(report,ensure_ascii=False,indent=2));print(json.dumps({k:v for k,v in report.items() if k!='label_counts'},ensure_ascii=False,indent=2));print('ANNOTATION_PROBE_COMPLETE')
